# Run your first federated job

Complete [setup](setup.ipynb) first. You will define a two-client, three-round FedAvg recipe, run it, and read its evaluation summary.

`FedAvgRecipe` supplies the standard server workflow and client execution setup. `SimEnv` runs the server and both sites locally. The same client script executes at each site, but the site name selects different local samples.

In [ ]:
import os
import sys
from pathlib import Path

# Make notebook shell commands use the selected kernel's environment.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")

lesson = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "running_pytorch_fl_job.ipynb").is_file())
code = lesson / "code"
assert (code / "job.py").is_file(), f"Keep this notebook beside its code directory: {code}"
print("Example code:", code)

## Define and run a recipe

These are the main API calls in `code/job.py`:

```python
from model import create_model
from nvflare.app_opt.pt.recipes.fedavg import FedAvgRecipe
from nvflare.recipe import SimEnv, add_final_global_evaluation

recipe = FedAvgRecipe(
    name="chapter1-basic",
    model=create_model(),
    min_clients=2,
    num_rounds=3,
    train_script="client.py",
)
add_final_global_evaluation(recipe)
run = recipe.execute(SimEnv(num_clients=2))
print("Results:", run.get_result())
```

`FedAvgRecipe` configures federated averaging and names the client training script. `add_final_global_evaluation()` adds evaluation of the saved model after training. `recipe.execute()` runs the recipe in `SimEnv`, and `run.get_result()` prints the summary and returns the results location.

## Run the example

The next cell runs `job.py` with the same command you use in a terminal. The initialization cell makes `python` use the notebook kernel's environment.

Expect three training rounds followed by a **Run Summary**. Its **Model evaluation** table shows the saved global model's accuracy at each site. The summary also gives the model and result locations.

In [ ]:
%cd {code}
!python job.py

## Interpret the result

During training, each client reports `accuracy` for the **received global model before local training**. The server aggregates that metric alongside the returned locally trained weights. That metric therefore does not evaluate the newly aggregated weights from the same round.

`add_final_global_evaluation(recipe)` explicitly adds evaluation after training. The `SRV_FL_global_model.pt` column in the summary measures the saved final aggregate on each site's evaluation data. Additional entries may describe the selected best model; do not confuse those with the final model.

The dataset is intentionally easy. Better accuracy on these generated images verifies the mechanics, not performance on CIFAR-10 or private data. The maintained [Hello PyTorch example](../../../../../hello-world/hello-pt/README.md) offers an optional CIFAR-10 path when you are ready for a larger dataset.

**Try it:** add `--num_rounds 1` to the command in the run cell and run it again. Compare final evaluation with three rounds. Explain why the final score cannot be inferred solely from the training-round metric.

Next: [convert local training to federated training](../01.2_convert_deep_learning_to_federated_learning/convert_dl_to_fl.ipynb).